In [1]:
# импортируем функцию для поднятия сессии и для отображения занятой памяти
import sys
import os

from functools import reduce
import yaml
from itertools import chain
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from dateutil.relativedelta import relativedelta
import datetime
from collections import defaultdict
import pyspark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import MinMaxScaler

from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import MapType, StringType, IntegerType, DoubleType, ByteType, LongType
from pyspark.sql.functions import from_json
from pyspark.sql import SparkSession
from pyspark import SparkConf

from dataclasses import dataclass
from IPython.display import display, clear_output
from typing import List, Union, Callable
import subprocess
import time

from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../')
sys.path.append('../../')
sys.path.append('../../../')
sys.path.append('../../../../')
sys.path.insert(0, "/home/datalab/nfs/zaripov/avatar_fm")
sys.path.insert(0, "/home/datalab/nfs/bogachev/")

import pyspark
from pyspark.sql import functions as F, types as T
from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import SparkSession
from pyspark import SparkConf
from tools.spark_session import create_spark_session

import time
import datetime

from avatar_fm.avatar.preprocessing.spark.pipeline import TabularPreprocessor

In [2]:
spark = create_spark_session(app_name='fmlib-bebra-selection', executor_instances=10)

Setting spark.hadoop.yarn.resourcemanager.principal to 23919316_omega-sbrf-ru
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Статус: OK
Spark UI: https://ci04139341-prom-datalabpro.apps.prom-terra000035-ias.ocp.ca.sbrf.ru/ci04139341-p-8713-avatar/datalabpro/jserver/proxy/4040/jobs/


In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
# (
#     spark.read.parquet(
#         "/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc"
#     )
#     .write
#     .partitionBy("split_type")
#     .mode('overwrite')
#     .format('parquet')
#     .option('compression', 'snappy')
#     .parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc_buff")
# )

In [5]:
# (
#     spark.read.parquet(
#         "/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_sms"
#     )
#     .write
#     .partitionBy("split_type")
#     .mode('overwrite')
#     .format('parquet')
#     .option('compression', 'snappy')
#     .parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_sms_buff")
# )

In [6]:

# train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/rusakov/feature_selection_benchmark/benchmark_datasets/product_name=tdbase_response/split_type=train")
# print(1)
# train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/rusakov/feature_selection_benchmark/benchmark_datasets/product_name=pd_cc/split_type=train")
# print(1)

# train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc/split_type=train")

# print(1)


In [2]:
try :
    spark.stop()
except:
    pass
    
spark = create_spark_session(app_name='fmlib-feature-selection', executor_instances=10)

%load_ext autoreload
%autoreload 2


train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/rusakov/feature_selection_benchmark/benchmark_datasets/product_name=pd_cc/split_type=train")
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/features_bio.yaml", "r") as f:
    feat_cols = yaml.safe_load(f)

from fmlib.feature_selection import (
    FeatureSchema,
    FeatureSelectionConfig,
    FeatureSelectionPipeline,
)
config = FeatureSelectionConfig.from_yaml(
 
    "/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/big_c/other.yaml"
)


pipeline = FeatureSelectionPipeline(config)
result = pipeline.fit_select(
    spark=spark,
    datasets={
        "train": train_df,
        # "valid": valid_df,
        # "test": test_df,
    }, 
    schema=FeatureSchema(
        categorical=feat_cols["cat_cols"],
        continuous=feat_cols["num_cols"],
        target="target_attr_1",
        task_type="binary_classification",
        time="month_part",
        # split="month_part",
        fold=None,
        id_columns=feat_cols["id_cols"],
    ),
    output_dir="pd/"
)
result.dropped_features


result.save("pd.json")

Setting spark.hadoop.yarn.resourcemanager.principal to 23919316_omega-sbrf-ru
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Статус: OK
Spark UI: https://ci04139341-prom-datalabpro.apps.prom-terra000035-ias.ocp.ca.sbrf.ru/ci04139341-p-8713-avatar/datalabpro/jserver/proxy/4040/jobs/


/home/datalab/nfs/bogachev/venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-10 10:18:27 ERROR TaskSchedulerImpl:76 - Lost executor 43 on spark-jlab-2244152-t5p3oql9y7rh-exec-43.spark-executors.ci04139341-p-8713-avatar.svc.cluster.local: The executor with id 43 was deleted by a user or the framework.
2026-08-10 10:18:42 ERROR TaskSchedulerImpl:76 - Lost executor 34 on spark-jlab-2244152-t5p3oql9y7rh-exec-34.spark-executors.ci04139341-p-8713-avatar.svc.cluster.local: The executor with id 34 was deleted by a user or the framework.
2026-08-10 10:23:40 ERROR TaskSchedulerImpl:76 - Lost executor 9 on spark-jlab-2244152-t5p3oql9y7rh-exec-9.spark-executors.ci04139341-p-8713-avatar.svc.cluster.local: The executor with id 9 was deleted by a user or the framework.
2026-08-10 11:02:59 ERROR TaskSchedu

ExecutionError: LightGbmSelector failed during feature selection. Root cause: An error occurred while calling o199337.getResult.
: org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:98)
	at org.apache.spark.security.SocketAuthServer.getResult(SocketAuthServer.scala:94)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEn

In [ ]:
# spark = create_spark_session(app_name='fmlib-feature-selection', executor_instances=10)

# %load_ext autoreload
# %autoreload 2
try :
    
    train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/zaripov/feature_selection_benchmark/product_name=sa_response_erkc/split_type=train")
    with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/features_bio.yaml", "r") as f:
        feat_cols = yaml.safe_load(f)
    
    from fmlib.feature_selection import (
        FeatureSchema,
        FeatureSelectionConfig,
        FeatureSelectionPipeline,
    )
    config = FeatureSelectionConfig.from_yaml(
     
        "/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/big_c/other.yaml"
    )
    
    pipeline = FeatureSelectionPipeline(config)
    result = pipeline.fit_select(
        spark=spark,
        datasets={
            "train": train_df,
            # "valid": valid_df,
            # "test": test_df,
        }, 
        schema=FeatureSchema(
            categorical=feat_cols["cat_cols"],
            continuous=feat_cols["num_cols"],
            target="target_attr_1",
            task_type="binary_classification",
            time="month_part",
            # split="month_part",
            fold=None,
            id_columns=feat_cols["id_cols"],
        ),
        output_dir="erkc/"
    )
    result.dropped_features
    
    result.save("erkc.json")
except:
    pass

In [ ]:

spark = create_spark_session(app_name='fmlib-feature-selection', executor_instances=10)

%load_ext autoreload
%autoreload 2
try :
    train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/rusakov/feature_selection_benchmark/benchmark_datasets/product_name=tdbase_response/split_type=train")
    with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/features_bio.yaml", "r") as f:
        feat_cols = yaml.safe_load(f)
    
    from fmlib.feature_selection import (
        FeatureSchema,
        FeatureSelectionConfig,
        FeatureSelectionPipeline,
    )
    config = FeatureSelectionConfig.from_yaml(
     
        "/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/big_c/other.yaml"
    )
    
    pipeline = FeatureSelectionPipeline(config)
    result = pipeline.fit_select(
        spark=spark,
        datasets={
            "train": train_df,
            # "valid": valid_df,
            # "test": test_df,
        }, 
        schema=FeatureSchema(
            categorical=feat_cols["cat_cols"],
            continuous=feat_cols["num_cols"],
            target="target_attr_1",
            task_type="binary_classification",
            time="month_part",
            # split="month_part",
            fold=None,
            id_columns=feat_cols["id_cols"],
        ),
        output_dir="td/"
    )
    result.dropped_features
    
    result.save("td.json")
except:
    pass

In [ ]:


spark = create_spark_session(app_name='abban', executor_instances=10)



train_df = spark.read.parquet("/user/team/team_ai_avatar/ds/zaripov/sa_sms_train_sample")
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/features_bio.yaml", "r") as f:
    feat_cols = yaml.safe_load(f)

import json
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_correlation_results.json", "r") as f:
    a = json.load(f)
with open("/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/sms/statistics_correlation_results.json", "r") as f:
    b = json.load(f)
c =  a["dropped_features"]+b["dropped_features"]

l  =  [ i["feature"] for i in c]

train_df = train_df.drop(*l)
    


from fmlib.feature_selection import (
    FeatureSchema,
    FeatureSelectionConfig,
    FeatureSelectionPipeline,
)
config = FeatureSelectionConfig.from_yaml(
 
    "/home/datalab/nfs/bogachev/sber-amazme-fmlib/examples/big_c/sms.yaml"
)
pipeline = FeatureSelectionPipeline(config)
result = pipeline.fit_select(
    spark=spark,
    datasets={
        "train": train_df,
        # "valid": valid_df,
        # "test": test_df,
    }, 
    schema=FeatureSchema(
        categorical=[i for i in feat_cols["cat_cols"] if i not in l],
        continuous=[i for i in feat_cols["num_cols"] if i not in l],
        target="target_attr_1",
        task_type="binary_classification",
        time="month_part",
        # split="month_part",
        fold=None,
        id_columns=feat_cols["id_cols"],
    ),
    output_dir="sms/"
)
result.dropped_features


result.save("sms.json")

[Stage 345:====================================================>  (40 + 1) / 42]

In [ ]:
spark.stop()

In [4]:
.


SyntaxError: invalid syntax (343737671.py, line 1)

In [7]:
c

[{'feature': 'bki_response_evt_attr_929',
  'stage': 'statistics',
  'method': 'null_rate',
  'reason': 'high_null_rate',
  'value': 1.0,
  'threshold': 0.95,
  'keep': False},
 {'feature': 'bki_response_evt_attr_920',
  'stage': 'statistics',
  'method': 'null_rate',
  'reason': 'high_null_rate',
  'value': 1.0,
  'threshold': 0.95,
  'keep': False},
 {'feature': 'bki_response_evt_attr_1015',
  'stage': 'statistics',
  'method': 'null_rate',
  'reason': 'high_null_rate',
  'value': 0.9818855500798477,
  'threshold': 0.95,
  'keep': False},
 {'feature': 'bki_response_evt_attr_248',
  'stage': 'statistics',
  'method': 'null_rate',
  'reason': 'high_null_rate',
  'value': 1.0,
  'threshold': 0.95,
  'keep': False},
 {'feature': 'bki_response_evt_attr_941',
  'stage': 'statistics',
  'method': 'null_rate',
  'reason': 'high_null_rate',
  'value': 1.0,
  'threshold': 0.95,
  'keep': False},
 {'feature': 'bki_response_evt_attr_1070',
  'stage': 'statistics',
  'method': 'null_rate',
  'rea